<a href="https://colab.research.google.com/github/eiad911/Learning-Git/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — ML Task Framing

## Refresh / Content Opportunity Scoring

This notebook frames the chosen Search Intelligence lane as a concrete ML problem. The goal is to rank pages by their priority for content refresh review.

## 1. My Lane as an ML Task

**Lane:** Refresh / Content Opportunity Scoring

**Task type:** Ranking / Scoring

The goal is to assign each page a priority score for content refresh. Pages can then be ranked from highest to lowest priority so that a content team can review the most important opportunities first.

The output is decision support, not proof that a page will definitely decline.

## 2. Target or Proxy

The target/proxy is whether a page is declining, based on its observed trend direction.

For the starter dataset:

`trend_direction == "down"`

This provides an observable outcome for learning which pages are more likely to need refresh review.

The final system must avoid using information that would only be known after the outcome, to prevent leakage.

## 3. Success Metric

The main success metric is **Precision@50**.

It answers: of the 50 pages ranked highest for refresh review, how many are actually declining?

This matches the real decision because a content team has limited time and needs a useful shortlist to review first.

In [5]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/eiad911/ml-1st-weak.git"
REPO_DIR = "/content/ml-1st-weak"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())
print("Files:", os.listdir()[:10])

Current directory: /content/ml-1st-weak
Files: ['scripts', '.github', 'AGENTS.md', 'submission', 'outputs', '.gitignore', 'SETUP.md', 'skills', 'GUIDE.md', 'notebooks']


In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Number of pages:", len(df))

df[[
    "trend_direction",
    "search_volume",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "days_since_last_update"
]].head(10)

Dataset shape: (30000, 44)
Number of pages: 30000


,trend_direction,search_volume,impressions_90d,avg_position,ctr,word_count,days_since_last_update
0,down,10.0,3803,10.6,0.76,3221.0,20
1,down,90.0,15320,20.3,0.05,2481.0,25
2,down,0.0,12581,36.5,0.09,3515.0,20
3,stable,10.0,11751,6.2,0.49,NaN,22
4,down,0.0,19140,44.0,0.13,2803.0,14
5,down,720.0,3970,8.5,0.03,3080.0,20
6,down,0.0,20,7.0,0.00,3059.0,20
7,stable,590.0,1724,21.2,0.06,NaN,22
8,down,0.0,32574,46.0,0.09,3807.0,20
9,down,0.0,1240,4.9,0.16,NaN,104


## 4. The Unit of Analysis

The unit of analysis is **one web page**.

Each row represents one page and contains observable search, engagement, and content characteristics for that page.

The model will therefore produce one refresh-priority score for each page.

In [7]:
# Sketch the target/proxy column
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Target distribution:")
print(df["is_declining"].value_counts().sort_index())

df[[
    "trend_direction",
    "is_declining",
    "impressions_90d",
    "avg_position",
    "ctr"
]].head(10)

Target distribution:
is_declining
0    13738
1    16262
Name: count, dtype: int64


,trend_direction,is_declining,impressions_90d,avg_position,ctr
0,down,1,3803,10.6,0.76
1,down,1,15320,20.3,0.05
2,down,1,12581,36.5,0.09
3,stable,0,11751,6.2,0.49
4,down,1,19140,44.0,0.13
5,down,1,3970,8.5,0.03
6,down,1,20,7.0,0.00
7,stable,0,1724,21.2,0.06
8,down,1,32574,46.0,0.09
9,down,1,1240,4.9,0.16


### Target interpretation

`is_declining = 1` means the page is observed as declining in the starter data.

`is_declining = 0` means the page is not in the `down` trend category.

This is a proxy for refresh opportunity, not proof that refreshing a page will cause traffic to increase.

## 5. Why ML Beats a Fixed Rule

A fixed rule might say that pages should be refreshed when they are old and have high impressions. This is useful as a baseline, but it captures only a small number of conditions.

Machine learning can combine multiple signals such as impressions, average position, CTR, content age, update recency, and word count. It can learn patterns and interactions between these signals that are difficult to capture with one manually written rule.

The goal is not to replace human judgment. The model provides a ranked shortlist that helps the content team decide which pages deserve attention first.

## 6. Self-Check

- [x] Chosen lane: Refresh / Content Opportunity Scoring
- [x] Defined the ML task type
- [x] Defined the target/proxy
- [x] Defined the success metric
- [x] Explained the unit of analysis
- [x] Created the target column
- [x] Explained why ML can improve on a fixed rule
- [x] Connected the output to a real content action

**Conclusion:** This is a page-level ranking/scoring problem where the output supports content refresh prioritization.